<a href="https://colab.research.google.com/github/Meghnashankr23/3DVSS_June2026/blob/main/Copy_of_LoRa_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! unzip /content/output.zip

In [ ]:
!unzip /content/output_filename.zip

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

# ================= CONFIGURATION =================
train_dir = "train"              # Your source folder
mask_path = "semantic_mask.png"  # Your mask
target_size = (768,768)

# OUTPUT CONFIGURATION
# We save metadata.csv INSIDE the folder so the training script finds it automatically
dir_palm = "dataset_palm"
dir_back = "dataset_back"
csv_palm = os.path.join(dir_palm, "metadata.csv")
csv_back = os.path.join(dir_back, "metadata.csv")
# =================================================

# 1. Load and Prepare Mask
mask_img = cv2.imread(mask_path)
mask_img = cv2.resize(mask_img, target_size, interpolation=cv2.INTER_NEAREST)

# OpenCV loads images as BGR (Blue, Green, Red)
# Channel 0 = Blue
# Channel 1 = Green
# Channel 2 = Red

# CORRECTED LOGIC:
# Blue Channel > 100 = Back of Hand
mask_back = (mask_img[:, :, 0] > 100).astype(np.uint8) * 255
# Red Channel > 100 = Palm
mask_palm = (mask_img[:, :, 2] > 100).astype(np.uint8) * 255

os.makedirs(dir_palm, exist_ok=True)
os.makedirs(dir_back, exist_ok=True)

data_palm = []
data_back = []

print("Splitting dataset into Palm and Back...")

for root, dirs, files in os.walk(train_dir):
    for filename in files:
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):

            # Paths & Info
            file_path = os.path.join(root, filename)
            folder_name = os.path.basename(root)      # Skin Type (e.g., "white")
            file_base = os.path.splitext(filename)[0] # Variation (e.g., "young with glittery nails")

            if root == train_dir: continue

            # Clean text for the PROMPT (Remove underscores, keep spaces)
            skin = folder_name.replace("_", " ")
            var = file_base.replace("_", " ")

            # Clean text for the FILENAME (Optional: replace spaces with underscores to be safe)
            # If you prefer keeping spaces in filenames, you can remove the .replace below
            safe_var = file_base.replace(" ", "_")
            safe_skin = folder_name.replace(" ", "_")

            try:
                # Load & Resize
                img = cv2.imread(file_path)
                img = cv2.resize(img, target_size, interpolation=cv2.INTER_LANCZOS4)

                # --- 1. CREATE PALM DATA ---
                img_palm = cv2.bitwise_and(img, img, mask=mask_palm)

                # Naming convention: white_young_with_glittery_nails_palm.png
                save_name_palm = f"{safe_skin}_{safe_var}_palm.png"
                save_path_palm = os.path.join(dir_palm, save_name_palm)
                cv2.imwrite(save_path_palm, img_palm)

                # Metadata: We only store the FILENAME (not the full path)
                data_palm.append({
                    "file_name": save_name_palm,
                    "text": f"photo of a human palm, {skin}, {var}, smooth skin, lines, high detail, 8k, uv layout"
                })

                # --- 2. CREATE BACK DATA ---
                img_back = cv2.bitwise_and(img, img, mask=mask_back)

                save_name_back = f"{safe_skin}_{safe_var}_back.png"
                save_path_back = os.path.join(dir_back, save_name_back)
                cv2.imwrite(save_path_back, img_back)

                # Metadata
                data_back.append({
                    "file_name": save_name_back,
                    "text": f"photo of the back of a human hand, {skin}, {var}, pores, wrinkles, knuckles, veins, 8k, uv layout"
                })

            except Exception as e:
                print(f"Error on {filename}: {e}")

# Save CSVs
pd.DataFrame(data_palm).to_csv(csv_palm, index=False)
pd.DataFrame(data_back).to_csv(csv_back, index=False)

print(f"✅ Done.")
print(f"   Palm dataset saved to: {dir_palm} (with metadata.csv inside)")
print(f"   Back dataset saved to: {dir_back} (with metadata.csv inside)")

In [ ]:
!pip install diffusers transformers accelerate bitsandbytes

In [ ]:
!pip install git+https://github.com/huggingface/diffusers.git

In [ ]:
!wget https://raw.githubusercontent.com/huggingface/diffusers/v0.36.0/examples/text_to_image/train_text_to_image_lora.py

In [ ]:
import os
import shutil

# Define your paths
folders = [
    ("dataset_palm", "metadata_palm.csv"),
    ("dataset_back", "metadata_back.csv")
]

for folder_name, csv_name in folders:
    # Check if the folder and CSV exist
    if os.path.exists(folder_name) and os.path.exists(csv_name):
        destination = os.path.join(folder_name, "metadata.csv")

        # Move and Rename
        shutil.move(csv_name, destination)
        print(f"✅ Fixed: Moved '{csv_name}' to '{destination}'")
    else:
        print(f"⚠️ Warning: Could not find '{folder_name}' or '{csv_name}'. Check if they exist.")

print("\nReady to train!")

In [ ]:
import pandas as pd
import os

# The folders we need to fix
folders = ["dataset_palm", "dataset_back"]

for folder in folders:
    csv_path = os.path.join(folder, "metadata.csv")

    if os.path.exists(csv_path):
        print(f"Fixing paths in {csv_path}...")

        # Read the CSV
        df = pd.read_csv(csv_path)

        # Function to strip the folder name from the path
        def clean_path(path):
            # Normalize slashes just in case
            path = path.replace("\\", "/")
            # If the path starts with "dataset_palm/", remove it
            if path.startswith(f"{folder}/"):
                return path.replace(f"{folder}/", "")
            # Fallback: Just return the filename itself
            return os.path.basename(path)

        # Apply the fix
        df['file_name'] = df['file_name'].apply(clean_path)

        # Save it back (overwrite)
        df.to_csv(csv_path, index=False)

        print(f"✅ Success! First entry is now: {df.iloc[0]['file_name']}")
    else:
        print(f"❌ Error: Could not find {csv_path}")

print("\nPaths are clean. You can run training now.")

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="SG161222/Realistic_Vision_V5.1_noVAE" \
  --train_data_dir="dataset_palm" \
  --caption_column="text" \
  --resolution=768 \
  --random_flip \
  --train_batch_size=1 \
  --num_train_epochs=350 \
  --learning_rate=1e-04 \
  --output_dir="lora_palm_v1" \
  --mixed_precision="fp16" \
  --checkpointing_steps=1000 \
  --resume_from_checkpoint="latest"

In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="SG161222/Realistic_Vision_V5.1_noVAE" \
  --train_data_dir="dataset_back" \
  --caption_column="text" \
  --resolution=768 \
  --random_flip \
  --train_batch_size=1 \
  --num_train_epochs=350 \
  --learning_rate=1e-04 \
  --output_dir="lora_back_v1" \
  --mixed_precision="fp16" \
 --resume_from_checkpoint="latest" \
  --checkpointing_steps=1000

In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

# ================= CONFIGURATION =================
base_model = "SG161222/Realistic_Vision_V5.1_noVAE"
path_lora_back = "/content/back"
path_lora_palm = "/content/palm"

# INPUT MAPS (Ensure these exist!)
path_depth  = "/content/uv_depth.png"   # Grayscale Depth Map
path_normal = "/content/controlnet_uv_normal_ring.png"  # Purple Normal Map
mask_path   = "semantic_mask.png"

# RESOLUTION SETTING (Locked to 768)
target_size = (768, 768)
# =================================================

# 1. Load Multi-ControlNet Pipeline
print("Loading models...")
# Load Depth and Normal ControlNets
cn_depth  = ControlNetModel.from_pretrained("lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16)
cn_normal = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_normalbae", torch_dtype=torch.float16)

# Pass them as a list: [Depth, Normal]
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model,
    controlnet=[cn_depth, cn_normal],
    torch_dtype=torch.float16
).to("cuda")
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# 2. Prepare Inputs
print(f"Preparing inputs at {target_size}...")

def load_image(path, mode="RGB"):
    # Resize to target_size (768x768) to match generation output
    return Image.open(path).convert(mode).resize(target_size, Image.Resampling.LANCZOS)

# Load Control Images
img_depth  = load_image(path_depth, "RGB")
img_normal = load_image(path_normal, "RGB")
control_images = [img_depth, img_normal] # List matching the model order [Depth, Normal]

# Load Mask (Nearest Neighbor to keep sharp edges)
mask_raw = Image.open(mask_path).convert("RGB").resize(target_size, Image.Resampling.NEAREST)
mask_arr = np.array(mask_raw)

# Create binary masks (Blue=Back, Red=Palm)
# Note: In PIL RGB, Red is index 0, Blue is index 2.
mask_palm_final = Image.fromarray((mask_arr[:,:,0] > 100).astype(np.uint8) * 255).convert("L")
mask_back_final = Image.fromarray((mask_arr[:,:,2] > 100).astype(np.uint8) * 255).convert("L")

def generate_perfect_hand(skin, variation):
    print(f"--- Generative Process: {skin} {variation} ---")

    # Common Inference Settings
    # We set scales: Depth=1.0 (Strong structure), Normal=0.8 (Texture details without noise)
    gen_kwargs = {
        "image": control_images,
        "height": target_size[0], # <--- CRITICAL FIX: 768
        "width": target_size[1],  # <--- CRITICAL FIX: 768
        "num_inference_steps":30,
        "guidance_scale":3,
        "controlnet_conditioning_scale": [0.8, 0.5]
    }

    # --- STEP A: GENERATE PALM ---
    print("1. Generating Palm...")
    pipe.unload_lora_weights()
    pipe.load_lora_weights(path_lora_palm)

    prompt_palm = f"photo of a human palm, {skin}, {variation}, smooth skin, lines, hyper-realistic, 8k, uv layout"

    image_palm = pipe(prompt_palm, **gen_kwargs).images[0]

    # --- STEP B: GENERATE BACK ---
    print("2. Generating Back...")
    pipe.unload_lora_weights()
    pipe.load_lora_weights(path_lora_back)

    prompt_back = f"photo of the back of a human hand, {skin}, {variation}, pores, wrinkles, knuckles, veins, 8k, uv layout"

    image_back = pipe(prompt_back, **gen_kwargs).images[0]

    # --- STEP C: STITCH TOGETHER ---
    print("3. Stitching...")

    # Start with black background (Now 768x768)
    final_comp = Image.new("RGB", target_size, (0, 0, 0))

    # Paste Palm using Palm Mask
    # NOW: Image is 768, Mask is 768. It will work.
    final_comp.paste(image_palm, (0,0), mask_palm_final)

    # Paste Back using Back Mask
    final_comp.paste(image_back, (0,0), mask_back_final)

    # Save
    filename = f"final_{skin.replace(' ','_')}_{variation.replace(' ','_')}.png"
    final_comp.save(filename)
    print(f"✅ Saved to {filename}")

# RUN
generate_perfect_hand("black", "old skin with glittery nails")
generate_perfect_hand("brown", "young skin with engagement ring")

In [ ]:
import zipfile
import os

zip_file_path = '/content/lora_back_v1.zip'
output_directory = '/content/' # Extract to the content directory

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_directory)

print(f"Successfully extracted {zip_file_path} to {output_directory}")

zip_file_path = '/content/lora_palm_v1.zip'
output_directory = '/content/' # Extract to the content directory

# Create the output directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(output_directory)

print(f"Successfully extracted {zip_file_path} to {output_directory}")

In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

# ================= CONFIGURATION =================
base_model = "SG161222/Realistic_Vision_V5.1_noVAE"
path_lora_back = "/content/lora_back_v1/pytorch_lora_weights.safetensors"
path_lora_palm = "/content/lora_palm_v1/pytorch_lora_weights.safetensors"

# INPUT MAPS (You need to upload these)
path_depth  = "/content/uv_depth.png"
path_normal = "/content/control_uv_normal.png"
mask_path   = "/content/semantic_mask.png"

# RESOLUTION SETTING (Locked to 768)
target_size = (768, 768)
# =================================================

# 1. Load Multi-ControlNet Pipeline
print("Loading models...")
# Load Depth and Normal ControlNets
cn_depth  = ControlNetModel.from_pretrained("lllyasviel/control_v11f1p_sd15_depth", torch_dtype=torch.float16)
cn_normal = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_normalbae", torch_dtype=torch.float16)

# Pass them as a list: [Depth, Normal]
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model,
    controlnet=[cn_depth, cn_normal],
    torch_dtype=torch.float16
).to("cuda")
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# 2. Prepare Inputs
print(f"Preparing inputs at {target_size}...")

def load_image(path, mode="RGB"):
    return Image.open(path).convert(mode).resize(target_size, Image.Resampling.LANCZOS)

# Load Control Images
img_depth  = load_image(path_depth, "RGB")
img_normal = load_image(path_normal, "RGB")
control_images = [img_depth, img_normal] # List matching the model order

# Load Mask (Nearest Neighbor to keep sharp edges)
mask_raw = Image.open(mask_path).convert("RGB").resize(target_size, Image.Resampling.NEAREST)
mask_arr = np.array(mask_raw)

# Create binary masks (Blue=Back, Red=Palm)
# Note: In PIL RGB, Red is index 0, Blue is index 2.
# Adjust indices [0] or [2] below if your mask colors are swapped.
mask_palm_final = Image.fromarray((mask_arr[:,:,0] > 100).astype(np.uint8) * 255).convert("L")
mask_back_final = Image.fromarray((mask_arr[:,:,2] > 100).astype(np.uint8) * 255).convert("L")

def generate_perfect_hand(skin, variation):
    print(f"--- Generative Process: {skin} {variation} ---")

    # Common Inference Settings
    # We set scales: Depth=1.0 (Strong structure), Normal=0.8 (Texture details without noise)
    gen_kwargs = {
        "image": control_images,
        "height": target_size[0],
        "width": target_size[1],
        "num_inference_steps":25,
        "guidance_scale":3.5,
        "controlnet_conditioning_scale": [0.6, 0.8]
    }

    # --- STEP A: GENERATE PALM ---
    print("1. Generating Palm...")
    pipe.unload_lora_weights()
    pipe.load_lora_weights(path_lora_palm)

    prompt_palm = f"photo of a human palm, {skin}, {variation}, smooth skin, lines, hyper-realistic, 8k, uv layout"

    image_palm = pipe(prompt_palm, **gen_kwargs).images[0]

    # --- STEP B: GENERATE BACK ---
    print("2. Generating Back...")
    pipe.unload_lora_weights()
    pipe.load_lora_weights(path_lora_back)

    prompt_back = f"photo of the back of a human hand, {skin}, {variation}, pores, wrinkles, knuckles, veins, 8k, uv layout"

    image_back = pipe(prompt_back, **gen_kwargs).images[0]

    # --- STEP C: STITCH TOGETHER ---
    print("3. Stitching...")

    # Start with black background
    final_comp = Image.new("RGB", target_size, (0, 0, 0))

    # Paste Palm using Palm Mask
    final_comp.paste(image_palm, (0,0), mask_palm_final)

    # Paste Back using Back Mask
    final_comp.paste(image_back, (0,0), mask_back_final)

    # Convert back image to numpy
    back_np = np.array(image_back)

    # Compute mean color of BACK image
    mean_color = tuple(np.mean(back_np.reshape(-1, 3), axis=0).astype(np.uint8))

    print(f"Background mean color: {mean_color}")

    # Create background using mean back color
    final_comp = Image.new("RGB", target_size, mean_color)

    # Paste Palm using Palm Mask
    final_comp.paste(image_palm, (0,0), mask_palm_final)

    # Paste Back using Back Mask
    final_comp.paste(image_back, (0,0), mask_back_final)

    # Save
    filename = f"final_{skin.replace(' ','_')}_{variation.replace(' ','_')}.png"
    final_comp.save(filename)

    print(f"✅ Saved to {filename}")


# RUN
generate_perfect_hand("white", "young skin")
generate_perfect_hand("darkbrown", "old skin with glittery nails ring")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!zip -r /content/LoRa_Textures.zip /content/lora_back_v1 /content/lora_palm_v1

In [ ]:
!pip install --upgrade torchao>=0.16.0

In [ ]:

!pip install "pyvista[jupyter]"
import pyvista as pv
pv.set_jupyter_backend('html')
pv.OFF_SCREEN = False

In [ ]:
texture = pv.read_texture('/content/final_darkbrown_old_skin_with_glittery_nails_ring.png')
plotter = pv.Plotter()
mesh = pv.read("/content/MANO_UV_right.obj")
plotter.add_mesh(mesh, texture=texture)
plotter.show()

### Convert Notebook to Python Script

To convert this Colab notebook (`.ipynb` file) into a Python script (`.py` file), you can use the `jupyter nbconvert` command.

First, you need to find the exact name of your notebook file. You can usually find this at the top of your browser tab or by listing the files in your current directory using `!ls`.

Then, replace `YourNotebookName.ipynb` in the command below with the actual name of your notebook.

In [ ]:
!pip list
